In [2]:
import os
import qsprpred

from qsprpred.data import QSPRDataset, RandomSplit
from qsprpred.data.descriptors.fingerprints import MorganFP
import pandas as pd
from qsprpred.data.descriptors.sets import RDKitDescs

In [4]:
def load_datasets(path):
    dataset = QSPRDataset.fromTableFile(
    filename=path,
    store_dir="dataset_outputs/A2AR/data",
    name="A2ARDataset",
    target_props=[{"name": "Y", "task": "SINGLECLASS", "th": [0.5]}],
    random_state=42,
    smiles_col = 'Drug',
    sep=','
    )
    dataset.prepareDataset(
    feature_calculators=[MorganFP(radius=2, nBits=1024)],
    recalculate_features=True,
    shuffle=False
    )
    from qsprpred.data.descriptors.sets import RDKitDescs
    
    rdkit_descs = RDKitDescs()
    
    dataset.addDescriptors([rdkit_descs])
    
    dataset.descriptorSets
    return dataset
    

In [5]:
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizerFast, RobertaForMaskedLM, DataCollatorWithPadding
from sklearn.base import BaseEstimator, TransformerMixin

class SMILESDataset(Dataset):
    def __init__(self, smiles, tokenizer, max_len=128):
        self.smiles = smiles
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        smile = self.smiles[idx]
        encoding = self.tokenizer(smile, truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')
        return encoding


class ChemBERTaTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, model_name="entropy/roberta_zinc_480m", max_len=128, batch_size=32, device=None):
        self.model_name = model_name
        self.max_len = max_len
        self.batch_size = batch_size
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = RobertaForMaskedLM.from_pretrained(self.model_name).to(self.device)
        self.tokenizer = RobertaTokenizerFast.from_pretrained(self.model_name, max_len=self.max_len)
        self.collator = DataCollatorWithPadding(self.tokenizer, padding=True, return_tensors='pt')
        self.embedding_dim = None  # bude nastaven po fit()

    def fit(self, X, y=None):
        smiles_dataset = SMILESDataset(X, self.tokenizer, max_len=self.max_len)
        dataloader = DataLoader(smiles_dataset, batch_size=1, collate_fn=self.collator)
        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch['input_ids'].squeeze(1).to(self.device)
                attention_mask = batch['attention_mask'].squeeze(1).to(self.device)
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
                embedding = outputs[1][-1]  # poslední hidden state
                self.embedding_dim = embedding.shape[-1]
                break
        return self

    def transform(self, X):
        self.model.eval()
        smiles_dataset = SMILESDataset(X, self.tokenizer, max_len=self.max_len)
        dataloader = DataLoader(smiles_dataset, batch_size=self.batch_size, collate_fn=self.collator)
        embeddings_list = []

        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch['input_ids'].squeeze(1).to(self.device)
                attention_mask = batch['attention_mask'].squeeze(1).to(self.device)
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
                full_embeddings = outputs[1][-1]
                embeddings = ((full_embeddings * attention_mask.unsqueeze(-1)).sum(1) / attention_mask.sum(-1).unsqueeze(-1))
                embeddings_list.append(embeddings)

        all_embeddings = torch.cat(embeddings_list, dim=0).cpu().numpy()
        column_names = [f"chemberta_{i}" for i in range(self.embedding_dim)]
        return pd.DataFrame(all_embeddings, columns=column_names)


In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

X1_all = load_datasets("A2AR/data/a2ar_train_1")

X2_all = load_datasets("A2AR/data/a2ar_val_1")

X3_all = load_datasets("A2AR/data/a2ar_test")

transformer = ChemBERTaTransformer()
X_train_emb = transformer.fit_transform(X1_all.df["Drug"])
X_val_emb = transformer.transform(X2_all.df["Drug"])
X_test_emb = transformer.transform(X3_all.df["Drug"])


/tmp/ipykernel_15700/195630644.py:17: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  smile = self.smiles[idx]


In [7]:
import pandas as pd
X1_all.X = X1_all.X.reset_index(drop=True)
X_train_emb = X_train_emb.reset_index(drop=True)
X1_all.X = pd.concat([X1_all.X, X_train_emb], axis = 1)

In [8]:
X2_all.X = X2_all.X.reset_index(drop=True)
X_val_emb = X_val_emb.reset_index(drop=True)
X2_all.X = pd.concat([X2_all.X, X_val_emb], axis = 1)
X3_all.X = X3_all.X.reset_index(drop=True)
X_test_emb = X_test_emb.reset_index(drop=True)
X3_all.X = pd.concat([X3_all.X, X_test_emb], axis = 1)

In [9]:
X1 = X1_all.X
y1 = X1_all.y
X2 = X2_all.X
y2 = X2_all.y
X3 = X3_all.X
y3 = X3_all.y

In [10]:
imp_mean = SimpleImputer(missing_values=pd.NA, strategy='mean')
X1 = imp_mean.fit_transform(X1)
X2 = imp_mean.transform(X2)
X3 = imp_mean.transform(X3)
scaler = StandardScaler()
scaler.fit(X1)
X1 = scaler.transform(X1)
X2 = scaler.transform(X2)
X3 = scaler.transform(X3)

In [12]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy=0.5, random_state=42)
X1, y1 = smote.fit_resample(X1, y1)
display(pd.DataFrame(X1))


,0,1,2,3,4,5,6,7,8,9,...,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001
0,-0.098367,-0.323209,-0.110594,-0.119876,-0.098367,-0.057831,-0.073798,-0.671586,-0.145865,-0.276977,...,-0.287995,0.475829,-1.886007,-0.019086,1.685174,-1.575181,-1.036012,0.372024,0.183762,-1.731507
1,-0.098367,-0.323209,-0.110594,-0.119876,-0.098367,-0.057831,-0.073798,-0.671586,-0.145865,-0.276977,...,-0.847985,-0.123727,1.436241,0.550994,-0.118027,0.899331,1.255971,0.138340,0.360206,-1.591655
2,-0.098367,-0.323209,-0.110594,-0.119876,-0.098367,-0.057831,-0.073798,1.489012,-0.145865,-0.276977,...,0.016308,-0.615823,-0.601442,-1.669977,1.384967,0.730146,1.281295,-1.245225,-1.563946,1.619037
3,-0.098367,-0.323209,-0.110594,-0.119876,-0.098367,-0.057831,-0.073798,-0.671586,-0.145865,-0.276977,...,0.648537,-1.275505,0.558520,0.940265,-0.989957,0.098584,-0.791973,0.670594,-0.313377,0.854122
4,-0.098367,-0.323209,-0.110594,-0.119876,-0.098367,-0.057831,-0.073798,-0.671586,-0.145865,-0.276977,...,-1.294618,0.290985,0.112522,1.052698,-0.938391,-0.994478,-0.979179,1.762197,-0.801921,-0.096545
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3460,-0.098367,-0.323209,-0.110594,-0.119876,-0.098367,-0.057831,-0.073798,1.489012,-0.145865,-0.276977,...,1.221856,0.039337,-0.345848,-0.257785,-0.808051,-1.189340,-0.115003,1.100244,1.236962,0.473864
3461,-0.098367,-0.323209,-0.110594,-0.119876,-0.098367,-0.057831,-0.073798,-0.671586,-0.145865,-0.276977,...,0.481551,1.128323,-1.079669,-0.538365,-0.174570,0.675272,0.816715,-0.223604,0.418711,-1.436246
3462,-0.098367,-0.323209,-0.110594,-0.119876,-0.098367,-0.057831,-0.073798,1.203936,-0.145865,-0.276977,...,1.169177,-0.821740,-0.880680,0.016980,-1.051404,-0.861372,1.060975,0.735636,2.086703,-0.020123
3463,-0.098367,-0.323209,-0.110594,-0.119876,-0.098367,-0.057831,-0.073798,-0.671586,-0.145865,-0.276977,...,1.154101,-1.817191,-1.057004,2.039169,-0.542305,-0.117311,2.184166,-0.026589,-1.009716,1.977249


In [13]:

# Přidejte cestu k vašemu lokálnímu repozitáři
import sys
import os

# Přidání cesty k lokálnímu repozitáři na začátek sys.path
sys.path.insert(0, '/home/ubuntu/Bakalarka/QSPRpred')

# Zkontrolujte, zda je cesta v sys.path
print(sys.path)

from importlib import reload

# Znovu proveďte import
from qsprpred.extra.gpu.models.neural_network import STFullyConnected
# Znovu načtěte modul, abyste zajistili, že je správně importován
reload(sys.modules['qsprpred.extra.gpu.models.neural_network'])

# Znovu proveďte import
from qsprpred.extra.gpu.models.neural_network import STFullyConnected

os.chdir('/home/ubuntu/Bakalarka/QSPRpred')
print(os.getcwd())


import sys
import importlib.util

# Přidání cesty k repozitáři do sys.path
sys.path.insert(0, '/home/ubuntu/Bakalarka/QSPRpred')

# Specifikujte cestu k souboru, který chcete importovat
module_path = '/home/ubuntu/Bakalarka/QSPRpred/qsprpred/extra/gpu/models/neural_network.py'
module_name = 'qsprpred.extra.gpu.models.neural_network'

# Načtěte modul z konkrétní cesty
spec = importlib.util.spec_from_file_location(module_name, module_path)
neural_network = importlib.util.module_from_spec(spec)
spec.loader.exec_module(neural_network)

# Nyní můžete používat třídu STFullyConnected
STFullyConnected = neural_network.STFullyConnected

['/home/ubuntu/Bakalarka/QSPRpred', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python311.zip', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/lib-dynload', '', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages']
/home/ubuntu/Bakalarka/QSPRpred
lol


In [14]:
from sklearn.model_selection import ParameterGrid
from torch.nn import functional as F
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score, matthews_corrcoef
import pandas as pd

def test_fun(dic,  X_train, y_train, X_test, y_test) -> pd.DataFrame:
    param_grid_t = ParameterGrid(dic)
    i = 0
    val_f1_t = []
    val_acc_t = []
    val_mcc_t = []
    param_len_t = len(param_grid_t)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device used:", device)
    for param in param_grid_t:
        i += 1
        print(i, '/', param_len_t)
        model_sts_t = STFullyConnected(n_dim=X_train.shape[1],  # počet vstupních neuronů (počet deskriptorů)
        n_class=1,  # regresní úloha (1 výstup)
        gpus=[],
        device=device,
        is_reg=False, **param)
        model_sts_t.fit(X_train, y_train)
        res = model_sts_t.predict(X_test)
        res = res >0.5
        val_f1_t.append(f1_score(res, y_test))
        val_acc_t.append(accuracy_score(res, y_test))
        val_mcc_t.append(matthews_corrcoef(res, y_test))
        print(param)
        print(f1_score(res, y_test))
        print(accuracy_score(res, y_test))
        print(matthews_corrcoef(res, y_test))
    my_df = pd.DataFrame(param_grid_t)
    my_df["F1"] = val_f1_t
    my_df["Acc"] = val_acc_t
    my_df["MCC"] = val_mcc_t
    my_df.sort_values(by="MCC", ascending=False, inplace=True)
    return my_df

In [22]:
from torch import optim
import torch
my_dict_ult = {
    "act_fun": [F.selu],
    "dropout_frac": [0, 0.15, 0.3, 0.45, 0.7],
    "patience": [75],
    "tol": [1e-5, 0],
    "weight_decay": [1e-4,],
    "n_epochs": [300],
    "neuron_layers": [[ 5000, 2500], [8192,4096, 2048, 1024, 512, 256], [4096, 2048, 1024, 512, 256], [4096, 2048, 1024, 512, 256, 128]]
    , "batch_size": [256]
    , "optimizer": [optim.AdamW] ,
    "lr": [1e-3, 1e-4,1e-5]
}
df_batch_ult = test_fun(my_dict_ult, X1, y1, X2, y2)
df_batch_ult

Device used: cuda
1 / 120
{'act_fun': <function selu at 0x7f64b0cb4ea0>, 'batch_size': 256, 'dropout_frac': 0, 'lr': 0.001, 'n_epochs': 300, 'neuron_layers': [5000, 2500], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 75, 'tol': 1e-05, 'weight_decay': 0.0001}
0.9669475048606611
0.9369592088998764
0.29486286054905725
2 / 120
{'act_fun': <function selu at 0x7f64b0cb4ea0>, 'batch_size': 256, 'dropout_frac': 0, 'lr': 0.001, 'n_epochs': 300, 'neuron_layers': [5000, 2500], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 75, 'tol': 0, 'weight_decay': 0.0001}
0.9669475048606611
0.9369592088998764
0.29486286054905725
3 / 120
{'act_fun': <function selu at 0x7f64b0cb4ea0>, 'batch_size': 256, 'dropout_frac': 0, 'lr': 0.001, 'n_epochs': 300, 'neuron_layers': [8192, 4096, 2048, 1024, 512, 256], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 75, 'tol': 1e-05, 'weight_decay': 0.0001}
0.9764181007010835
0.9542645241038319
0.22873349178001748
4 / 120
{'act_fun': <f

In [ ]:
from pathlib import Path

cesta = Path('qsprpred/extra/gpu/models/dataset_outputs/A2AR/tabs/')
pocet_souboru = sum(1 for f in cesta.iterdir() if f.is_file())
my_df.to_csv(f'qsprpred/extra/gpu/models/dataset_outputs/A2AR/tabs/A2ARbatch{}.csv')